# RFM Analysis — L0 Data Validation & L1 Customer Segmentation

Reproducible RFM notebook built on the standard field spec (`RFM_Fields_v2.xlsx`).

**How to run (Colab)**
1. Fill in the **Configuration** cell (data source, paths, snapshot date, reporting currency).
2. `Runtime → Run all`. When reading from Google Drive, approve the Drive access prompt.
3. Read the L0 report first. Critical issues stop the run; warnings are fixed automatically and logged.
4. Outputs (segments per customer, KPIs, DQ report, thresholds) are written to `OUTPUT_FOLDER`.

**Inputs** (CSV in Drive, CSV upload, or BigQuery tables)

| Table | Needed for | Status |
|---|---|---|
| `transactions` | L0, L1 | Required |
| `customers` | identity linking, New vs Reactivated, consent | Recommended |
| `fx_rates` | currency normalisation | Required if more than one currency |
| `items` | L0 reconciliation only (L3 later) | Optional here |

## 1. Configuration
Everything client-specific lives in this cell. Keep it unchanged between recurring runs of the same client.

In [ ]:
# ---------------- Data source ----------------
SOURCE = "drive"            # "drive" | "upload" | "bigquery" | "local"

# drive: folder in your Google Drive holding the CSVs (after mounting, My Drive = /content/drive/MyDrive)
DRIVE_FOLDER = "/content/drive/MyDrive/RFM/dummy_data"
# local: any folder path (used for testing outside Colab)
LOCAL_FOLDER = "./data"

FILE_NAMES = {               # file name per table; set to None if the client did not provide it
    "transactions": "transactions.csv",
    "customers":    "customers.csv",
    "fx_rates":     "fx_rates.csv",
    "items":        "items.csv",
}

# bigquery: fully qualified table ids; set to None if not provided
BQ_BILLING_PROJECT = "your-gcp-project"
BQ_TABLES = {
    "transactions": "your-gcp-project.rfm_dummy.transactions",
    "customers":    "your-gcp-project.rfm_dummy.customers",
    "fx_rates":     "your-gcp-project.rfm_dummy.fx_rates",
    "items":        "your-gcp-project.rfm_dummy.items",
}

# Optional renaming of client columns to spec names, per table: {"client_col": "spec_col"}
COLUMN_MAP = {
    "transactions": {},
    "customers": {},
    "fx_rates": {},
    "items": {},
}

# ---------------- Run settings ----------------
CLIENT_NAME = "Dummy Nordic Retail"
SNAPSHOT_DATE = "2026-11-01"     # Recency is counted to this date (exclusive). None = day after latest transaction
LOOKBACK_MONTHS = 24             # analysis window length
REPORTING_CURRENCY = "SEK"       # all monetary values converted to this currency

# Frequency: fixed bins (lower edges of score 1..5). Default: 1 | 2 | 3 | 4-5 | 6+
F_BIN_EDGES = [1, 2, 3, 4, 6]
# Recency and Monetary: quintiles. To reuse thresholds of an earlier run (L5), point to its rfm_thresholds.json
FROZEN_THRESHOLDS_PATH = None

# Outliers / B2B: customers above these quantiles are flagged
OUTLIER_QUANTILE = 0.999
OUTLIER_ACTION = "exclude"       # "exclude" = removed from scoring and reported separately | "keep"
RECONCILIATION_TOLERANCE = 0.01  # items vs transaction amount, relative difference

# Segment rules: evaluated top to bottom, first match wins.
# Available variables: R, F, M (scores 1-5), recency_days, frequency, monetary, has_prior_history (bool)
SEGMENT_RULES = [
    ("Champions",           "R >= 4 and F >= 4 and M >= 4"),
    ("Loyal High Value",    "R >= 3 and F >= 4 and M >= 4"),
    ("At Risk High Value",  "R <= 2 and F >= 3 and M >= 4"),
    ("New Customers",       "R >= 4 and F == 1 and not has_prior_history"),
    ("Reactivated",         "R >= 4 and F == 1 and has_prior_history"),   # remove line to fold into Potential Loyalists
    ("Potential Loyalists", "R >= 4"),
    ("Needs Attention",     "R == 3"),
    ("Hibernating",         "R <= 2"),
]
SEGMENT_INFO = {   # description and suggested action, written to the output
    "Champions":           ("Bought recently, often and spend the most.", "Reward, early access, referral; protect from discounting."),
    "Loyal High Value":    ("Frequent, high-spending customers, not quite as recent.", "Loyalty benefits, upsell premium ranges."),
    "At Risk High Value":  ("Used to buy often and spend a lot, but have not purchased for a long time.", "Win-back with personalised offers; highest revenue at stake."),
    "New Customers":       ("First-ever purchase, made recently.", "Onboarding flow, second-purchase incentive."),
    "Reactivated":         ("One recent purchase in the window, but with purchase history before it.", "Re-engage quickly to rebuild the habit."),
    "Potential Loyalists": ("Recent customers with a few purchases.", "Cross-sell, membership invitation."),
    "Needs Attention":     ("Average recency; starting to slip.", "Reminders, recommendations, limited-time offers."),
    "Hibernating":         ("Last purchase long ago, low frequency.", "Low-cost reactivation or suppress to save budget."),
}

# ---------------- Output ----------------
OUTPUT_FOLDER = None             # None = <source folder>/output (drive/local) or ./output
WRITE_TO_BIGQUERY = False        # also write the customer segment table to BigQuery
BQ_OUTPUT_TABLE = "your-gcp-project.rfm_output.customer_segments"

## 2. Setup

In [ ]:
import sys, os, re, json, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

IN_COLAB = "google.colab" in sys.modules
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Chart styling (validated categorical slots + sequential blue ramp)
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "text": "#0b0b0b", "muted": "#52514e",
     "grid": "#e6e5e1", "surface": "#fcfcfb"}
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

def style(fig, title, height=420):
    fig.update_layout(title=dict(text=title, x=0, font=dict(size=16, color=C["text"])),
                      font=dict(family="Arial", size=12, color=C["text"]),
                      paper_bgcolor=C["surface"], plot_bgcolor=C["surface"], height=height,
                      margin=dict(l=60, r=30, t=70, b=50), hoverlabel=dict(bgcolor="white"),
                      legend=dict(orientation="h", y=1.08, x=0))
    fig.update_xaxes(gridcolor=C["grid"], zeroline=False, linecolor=C["grid"])
    fig.update_yaxes(gridcolor=C["grid"], zeroline=False, linecolor=C["grid"])
    return fig

print("Running in Colab" if IN_COLAB else "Running locally")

## 3. Load data
Drive: mounts your Google Drive and reads the CSVs from `DRIVE_FOLDER`.
Upload: opens a file picker; upload the CSVs named as in `FILE_NAMES`.
BigQuery: authenticates your Google account and reads the tables in `BQ_TABLES`.

In [ ]:
def read_csv(path):
    # Everything as text first; types are parsed and validated in L0
    return pd.read_csv(path, dtype=str, keep_default_na=True)

def load_tables():
    raw = {}
    if SOURCE in ("drive", "local"):
        folder = DRIVE_FOLDER if SOURCE == "drive" else LOCAL_FOLDER
        if SOURCE == "drive":
            if not IN_COLAB:
                raise RuntimeError("SOURCE='drive' needs Colab. Use SOURCE='local' outside Colab.")
            from google.colab import drive
            drive.mount("/content/drive")
        if not os.path.isdir(folder):
            raise FileNotFoundError(f"Folder not found: {folder}")
        print("Files in folder:", sorted(os.listdir(folder)))
        for t, fn in FILE_NAMES.items():
            if fn and os.path.exists(os.path.join(folder, fn)):
                raw[t] = read_csv(os.path.join(folder, fn))
    elif SOURCE == "upload":
        from google.colab import files
        import io
        print("Select the CSV files:", [f for f in FILE_NAMES.values() if f])
        up = files.upload()
        for t, fn in FILE_NAMES.items():
            if fn in up:
                raw[t] = pd.read_csv(io.BytesIO(up[fn]), dtype=str)
    elif SOURCE == "bigquery":
        if IN_COLAB:
            from google.colab import auth
            auth.authenticate_user()
        from google.cloud import bigquery
        client = bigquery.Client(project=BQ_BILLING_PROJECT)
        for t, tid in BQ_TABLES.items():
            if tid:
                df = client.query(f"SELECT * FROM `{tid}`").to_dataframe()
                raw[t] = df.astype("string").astype(object).where(df.notna(), None)
    else:
        raise ValueError(f"Unknown SOURCE: {SOURCE}")

    for t in raw:
        raw[t] = raw[t].rename(columns=COLUMN_MAP.get(t, {}))
        raw[t].columns = [c.strip().lower() for c in raw[t].columns]
    return raw

raw = load_tables()
for t, df in raw.items():
    print(f"{t:<13} {len(df):>10,} rows  {df.shape[1]} columns")
if "transactions" not in raw:
    raise RuntimeError("The transactions table is required.")

## 4. L0 — Data validation
### 4.1 Field mapping against the spec

In [ ]:
SPEC = {  # table -> column -> (type, requirement)
    "transactions": {
        "transaction_id": ("string", "Required"), "transaction_date": ("date", "Required"),
        "hashed_email": ("hash", "Required*"), "customer_id": ("string", "Required*"),
        "transaction_amount": ("number", "Required"), "transaction_type": ("string", "Recommended"),
        "original_transaction_id": ("string", "Optional"), "basket_size": ("number", "Optional"),
        "currency": ("string", "Required for L2 / multi-currency"), "market": ("string", "Required for L2"),
        "medium": ("string", "Required for L4"), "store_id": ("string", "Optional"),
    },
    "customers": {
        "hashed_email": ("hash", "Required*"), "customer_id": ("string", "Required*"),
        "first_purchase_date": ("date", "Recommended"), "loyalty_member": ("bool", "Optional"),
        "marketing_consent": ("bool", "Optional"),
    },
    "items": {
        "transaction_id": ("string", "Required"), "item_sku": ("string", "Required"),
        "item_quantity": ("number", "Required"), "item_price": ("number", "Required"),
        "item_category": ("string", "Required"), "item_sub_category": ("string", "Optional"),
        "item_discount": ("number", "Optional"),
    },
    "fx_rates": {"month": ("string", "Required"), "currency": ("string", "Required"), "rate": ("number", "Required")},
}

# fx_rates: accept any "rate*" column name (e.g. rate_to_sek)
if "fx_rates" in raw and "rate" not in raw["fx_rates"].columns:
    rc = [c for c in raw["fx_rates"].columns if c.startswith("rate")]
    if rc:
        raw["fx_rates"] = raw["fx_rates"].rename(columns={rc[0]: "rate"})

mapping_rows = []
for t, cols in SPEC.items():
    present = set(raw[t].columns) if t in raw else set()
    for col, (typ, req) in cols.items():
        mapping_rows.append(dict(table=t, column=col, requirement=req,
                                 status=("table not provided" if t not in raw else "present" if col in present else "MISSING")))
    for col in sorted(present - set(cols)):
        mapping_rows.append(dict(table=t, column=col, requirement="not in spec", status="ignored"))
mapping = pd.DataFrame(mapping_rows)

tx_cols = set(raw["transactions"].columns)
critical_missing = [c for c in ["transaction_id", "transaction_date", "transaction_amount"] if c not in tx_cols]
if not ({"hashed_email", "customer_id"} & tx_cols):
    critical_missing.append("hashed_email or customer_id")
display(mapping.style.apply(lambda r: ["background-color:#fde2e1" if r.status == "MISSING" and r.requirement.startswith("Required")
                                        else "background-color:#fff4d6" if r.status == "MISSING" else "" for _ in r], axis=1))
if critical_missing:
    raise RuntimeError(f"CRITICAL: transactions is missing required columns: {critical_missing}")
for c in SPEC["transactions"]:
    if c not in raw["transactions"].columns:
        raw["transactions"][c] = None   # optional columns added empty so the code below runs

### 4.2 Data quality checks and cleaning
Each check records how many rows are affected and what the notebook does about it.
`critical` = needs the client to fix the export; `warning` = fixed automatically; `info` = for awareness.

In [ ]:
dq, clog = [], []
def issue(table, check, severity, rows, total, action):
    dq.append(dict(table=table, check=check, severity=severity, rows=int(rows),
                   share=(rows / total if total else 0), action=action))
def log(step, before, after):
    clog.append(dict(step=step, rows_before=before, rows_after=after, removed=before - after))

HASH_RE = re.compile(r"^[0-9a-f]{64}$")
def clean_hash(s, table):
    s = s.astype("string").str.strip()
    n = s.notna().sum()
    pii = s.str.contains("@", na=False)
    issue(table, "hashed_email contains plain email (PII)", "critical", pii.sum(), n, "Values removed; client must hash before sharing")
    upper = s.str.fullmatch(r"[0-9A-Fa-f]{64}", na=False) & ~s.str.fullmatch(r"[0-9a-f]{64}", na=False)
    issue(table, "hashed_email in uppercase", "warning", upper.sum(), n, "Lowercased")
    s = s.str.lower()
    bad = s.notna() & ~s.str.fullmatch(r"[0-9a-f]{64}", na=False)
    issue(table, "hashed_email not a 64-char SHA-256 hex", "warning", (bad & ~pii).sum(), n, "Set to empty; row falls back to customer_id")
    return s.where(~bad, pd.NA)

tx = raw["transactions"].copy()
N0 = len(tx)

# Duplicates
n = len(tx); dup = tx.duplicated()
issue("transactions", "exact duplicate rows", "warning", dup.sum(), n, "Removed")
tx = tx[~dup]; log("remove exact duplicates", n, len(tx))
n = len(tx); dup_id = tx.transaction_id.duplicated(keep="first")
issue("transactions", "same transaction_id, different content", "warning", dup_id.sum(), n, "First row kept")
tx = tx[~dup_id]; log("remove conflicting duplicate ids", n, len(tx))
issue("transactions", "transaction_id empty", "warning", tx.transaction_id.isna().sum(), len(tx), "Rows removed")
n = len(tx); tx = tx[tx.transaction_id.notna()]; log("remove empty transaction_id", n, len(tx))

# Dates
tx["transaction_date"] = pd.to_datetime(tx.transaction_date.str[:10], errors="coerce", format="%Y-%m-%d")
bad_date = tx.transaction_date.isna()
issue("transactions", "transaction_date missing or not YYYY-MM-DD", "warning", bad_date.sum(), len(tx), "Rows removed")
n = len(tx); tx = tx[~bad_date]; log("remove invalid dates", n, len(tx))

SNAP = pd.Timestamp(SNAPSHOT_DATE) if SNAPSHOT_DATE else tx.transaction_date.max() + pd.Timedelta(days=1)
WSTART = SNAP - pd.DateOffset(months=LOOKBACK_MONTHS)
fut = tx.transaction_date >= SNAP
old = tx.transaction_date < WSTART
issue("transactions", f"date on/after snapshot {SNAP.date()}", "warning" if fut.mean() < 0.01 else "critical", fut.sum(), len(tx), "Rows removed")
issue("transactions", f"date before window start {WSTART.date()}", "info", old.sum(), len(tx), "Outside window; removed")
n = len(tx); tx = tx[~fut & ~old]; log("keep analysis window", n, len(tx))

# Amounts
amt = pd.to_numeric(tx.transaction_amount, errors="coerce")
issue("transactions", "transaction_amount missing or non-numeric", "warning", amt.isna().sum(), len(tx), "Rows removed")
tx["transaction_amount"] = amt
n = len(tx); tx = tx[amt.notna()]; log("remove invalid amounts", n, len(tx))

# Transaction type
tt = tx.transaction_type.astype("string").str.strip().str.lower()
tt = tt.replace({"refund": "return", "sale": "purchase", "order": "purchase"})
if tt.isna().all():
    tt = pd.Series(np.where(tx.transaction_amount < 0, "return", "purchase"), index=tx.index)
    issue("transactions", "transaction_type not provided", "info", len(tx), len(tx), "Inferred from sign of amount")
else:
    unk = ~tt.isin(["purchase", "return"])
    issue("transactions", "transaction_type not purchase/return", "warning", unk.sum(), len(tx), "Inferred from sign of amount")
    tt = tt.where(~unk, np.where(tx.transaction_amount < 0, "return", "purchase"))
tx["transaction_type"] = tt
neg_p = (tx.transaction_type == "purchase") & (tx.transaction_amount < 0)
pos_r = (tx.transaction_type == "return") & (tx.transaction_amount > 0)
issue("transactions", "purchase with negative amount", "warning", neg_p.sum(), len(tx), "Treated as return")
issue("transactions", "return with positive amount", "warning", pos_r.sum(), len(tx), "Sign flipped to negative")
tx.loc[neg_p, "transaction_type"] = "return"
tx.loc[pos_r, "transaction_amount"] *= -1
zero = (tx.transaction_type == "purchase") & (tx.transaction_amount == 0)
issue("transactions", "purchase with zero amount", "info", zero.sum(), len(tx), "Kept (counts towards Frequency)")
ret = tx.transaction_type == "return"
issue("transactions", "return without original_transaction_id", "info",
      (ret & tx.original_transaction_id.isna()).sum(), ret.sum(), "Kept; reduces Monetary in its own month")

# Identifiers
tx["hashed_email"] = clean_hash(tx.hashed_email, "transactions")
tx["customer_id"] = tx.customer_id.astype("string").str.strip().replace("", pd.NA)

# Currency + FX
cur = tx.currency.astype("string").str.strip()
lower = cur.notna() & (cur != cur.str.upper())
issue("transactions", "currency not uppercase", "warning", lower.sum(), len(tx), "Uppercased")
cur = cur.str.upper()
miss = cur.isna()
if miss.any() and tx.market.notna().any():   # fill from the market's dominant currency
    mkt_cur = (pd.DataFrame({"m": tx.market, "c": cur}).dropna().groupby("m").c.agg(lambda s: s.mode().iat[0]))
    cur = cur.fillna(tx.market.map(mkt_cur))
issue("transactions", "currency missing", "warning", miss.sum(), len(tx),
      "Filled from the market's main currency" if tx.market.notna().any() else "Assumed reporting currency")
tx["currency"] = cur.fillna(REPORTING_CURRENCY)

if tx.currency.nunique() > 1 or tx.currency.iat[0] != REPORTING_CURRENCY:
    if "fx_rates" not in raw:
        raise RuntimeError("CRITICAL: several currencies found but no fx_rates table provided.")
    fx = raw["fx_rates"].copy()
    fx["currency"] = fx.currency.str.strip().str.upper()
    fx["rate"] = pd.to_numeric(fx.rate, errors="coerce")
    fx["month"] = fx.month.str[:7]
    fx = fx.dropna().drop_duplicates(["month", "currency"])
    tx["month"] = tx.transaction_date.dt.strftime("%Y-%m")
    tx = tx.merge(fx[["month", "currency", "rate"]], on=["month", "currency"], how="left")
    # fallback: nearest available month of the same currency
    if tx.rate.isna().any():
        fx_sorted = fx.sort_values("month")
        last_rate = fx_sorted.groupby("currency").rate.last()
        tx["rate"] = tx.rate.fillna(tx.currency.map(last_rate))
    tx.loc[tx.currency == REPORTING_CURRENCY, "rate"] = tx.loc[tx.currency == REPORTING_CURRENCY, "rate"].fillna(1.0)
    nofx = tx.rate.isna()
    issue("transactions", "no FX rate for currency", "critical" if nofx.mean() > 0.01 else "warning",
          nofx.sum(), len(tx), "Rows removed")
    n = len(tx); tx = tx[~nofx]; log("remove rows without FX rate", n, len(tx))
else:
    tx["rate"] = 1.0
tx["amount_rc"] = tx.transaction_amount * tx.rate     # amount in reporting currency

display(Markdown(f"**Analysis window:** {WSTART.date()} → {(SNAP - pd.Timedelta(days=1)).date()}  "
                 f"(snapshot {SNAP.date()}, {LOOKBACK_MONTHS} months) · **Reporting currency:** {REPORTING_CURRENCY}"))

### 4.3 Identity: linking email and customer ID
Each customer gets one `customer_key`: the hashed email where known, otherwise the email linked to the
customer ID (through the customers table or rows that carry both), otherwise the customer ID itself.

In [ ]:
cu = raw.get("customers")
if cu is not None:
    cu = cu.copy()
    cu["hashed_email"] = clean_hash(cu.get("hashed_email", pd.Series(index=cu.index, dtype="string")), "customers")
    cu["customer_id"] = cu.get("customer_id", pd.Series(index=cu.index, dtype="string")).astype("string").str.strip()

pairs = [tx[["customer_id", "hashed_email"]]]
if cu is not None:
    pairs.append(cu[["customer_id", "hashed_email"]])
pairs = pd.concat(pairs).dropna()
cid2email = pairs.groupby("customer_id").hashed_email.agg(lambda s: s.value_counts().index[0])
conflict = pairs.drop_duplicates().groupby("customer_id").hashed_email.nunique()
issue("identity", "customer_id linked to several emails", "info", (conflict > 1).sum(), len(conflict), "Most frequent email used")

linked = tx.hashed_email.isna() & tx.customer_id.map(cid2email).notna()
tx["customer_key"] = tx.hashed_email.fillna(tx.customer_id.map(cid2email))
tx["customer_key"] = tx.customer_key.fillna("cid:" + tx.customer_id)   # NA stays NA

ident = tx.customer_key.notna()
purch = tx.transaction_type == "purchase"
has_medium = tx.medium.notna().any()
if has_medium:
    ecom_anon = (tx.medium.str.lower() == "ecommerce") & ~ident
    issue("transactions", "ecommerce transaction without any identifier", "warning", ecom_anon.sum(),
          (tx.medium.str.lower() == "ecommerce").sum(), "Excluded from RFM (check the export)")
issue("transactions", "transaction without identifier (anonymous)", "info", (~ident).sum(), len(tx), "Excluded from RFM")
issue("identity", "rows linked via customer_id → email", "info", linked.sum(), len(tx), "Merged into the email's customer")

coverage = {
    "Purchases identified (share of transactions)": (ident & purch).sum() / purch.sum(),
    "Revenue identified (share of net revenue)": tx.loc[ident, "amount_rc"].sum() / tx.amount_rc.sum(),
}
if has_medium:
    rp = purch & (tx.medium.str.lower() == "retail")
    if rp.any():
        coverage["Retail purchases linked to a customer"] = (rp & ident).sum() / rp.sum()
    ep = purch & (tx.medium.str.lower() == "ecommerce")
    if ep.any():
        coverage["Ecommerce purchases linked to a customer"] = (ep & ident).sum() / ep.sum()
display(pd.DataFrame({"coverage": coverage}).style.format("{:.1%}"))

n = len(tx); txi = tx[ident].copy(); log("keep identified transactions", n, len(txi))

### 4.4 Items reconciliation (if items are provided)

In [ ]:
if "items" in raw:
    it = raw["items"].copy()
    it["item_price"] = pd.to_numeric(it.item_price, errors="coerce")
    orphan = ~it.transaction_id.isin(raw["transactions"].transaction_id)
    issue("items", "item line without matching transaction", "warning", orphan.sum(), len(it), "Ignored")
    issue("items", "item_price missing or non-numeric", "warning", it.item_price.isna().sum(), len(it), "Ignored in reconciliation")
    s = it.groupby("transaction_id").item_price.sum()
    rec = tx.set_index("transaction_id").transaction_amount.to_frame().join(s.rename("items_sum"), how="inner")
    diff = (rec.transaction_amount - rec.items_sum).abs() > RECONCILIATION_TOLERANCE * rec.transaction_amount.abs().clip(lower=1)
    issue("items", f"transaction_amount ≠ sum of item lines (>{RECONCILIATION_TOLERANCE:.0%})", "warning",
          diff.sum(), len(rec), "Transaction amount used for RFM; check VAT/shipping basis")
    no_items = ~tx.transaction_id.isin(it.transaction_id)
    issue("items", "transaction without item lines", "info", no_items.sum(), len(tx), "Relevant for L3")
else:
    print("Items table not provided — reconciliation skipped.")

### 4.5 Customer attributes and outliers

In [ ]:
# Per-customer metrics in the window
p = txi[txi.transaction_type == "purchase"]
r = txi[txi.transaction_type == "return"]
cust = p.groupby("customer_key").agg(
    last_purchase=("transaction_date", "max"), first_purchase_window=("transaction_date", "min"),
    frequency=("transaction_id", "nunique"), gross_revenue=("amount_rc", "sum"))
cust["returns"] = r.groupby("customer_key").amount_rc.sum().reindex(cust.index).fillna(0)
cust["monetary"] = cust.gross_revenue + cust.returns
cust["aov"] = cust.gross_revenue / cust.frequency
cust["recency_days"] = (SNAP - cust.last_purchase).dt.days
only_ret = set(r.customer_key) - set(cust.index)
issue("customers", "customer with returns but no purchase in window", "info", len(only_ret), len(cust) + len(only_ret), "Excluded")

issue("customers", "net revenue ≤ 0 after returns (fully returned)", "info", int((cust.monetary <= 0).sum()), len(cust), "Kept; scored M = 1")

# Enrichment from the customers table
cust["first_purchase_date"] = pd.NaT
for c in ["loyalty_member", "marketing_consent"]:
    cust[c] = pd.NA
if cu is not None:
    cu["customer_key"] = cu.hashed_email.fillna(cu.customer_id.map(cid2email)).fillna("cid:" + cu.customer_id)
    cu = cu[cu.customer_key.notna()]
    dupk = cu.customer_key.duplicated()
    issue("customers", "duplicate customer rows", "info", dupk.sum(), len(cu), "Earliest first_purchase_date kept")
    def to_bool(s):
        return s.astype("string").str.strip().str.lower().map({"true": True, "1": True, "yes": True, "y": True,
                                                                 "false": False, "0": False, "no": False, "n": False})
    agg = {}
    if "first_purchase_date" in cu:
        cu["first_purchase_date"] = pd.to_datetime(cu.first_purchase_date.str[:10], errors="coerce")
        agg["first_purchase_date"] = "min"
    for c in ["loyalty_member", "marketing_consent"]:
        if c in cu:
            cu[c] = to_bool(cu[c]); agg[c] = "max"
    enr = cu.groupby("customer_key").agg(agg) if agg else None
    if enr is not None:
        for c in enr.columns:
            cust[c] = enr[c].reindex(cust.index)
    matched = cust.index.isin(cu.customer_key)
    issue("customers", "RFM customers not found in customers table", "info", (~matched).sum(), len(cust),
          "No enrichment for these customers")
    later = cust.first_purchase_date > cust.first_purchase_window
    issue("customers", "first_purchase_date later than first purchase in data", "warning", later.sum(), len(cust),
          "Replaced by first purchase in data")
    cust.loc[later, "first_purchase_date"] = cust.loc[later, "first_purchase_window"]
else:
    print("Customers table not provided — New vs Reactivated uses the window only.")
cust["has_prior_history"] = (cust.first_purchase_date < cust.first_purchase_window).fillna(False).astype(bool)

# Outliers / B2B
f_cut = cust.frequency.quantile(OUTLIER_QUANTILE)
m_cut = cust.monetary.quantile(OUTLIER_QUANTILE)
# only flag when far above the bulk (avoids flagging normal top customers in small datasets)
f_cut = max(f_cut, cust.frequency.quantile(0.99) * 3)
m_cut = max(m_cut, cust.monetary.quantile(0.99) * 3)
cust["is_outlier"] = (cust.frequency > f_cut) | (cust.monetary > m_cut)
issue("customers", f"outlier / possible B2B (F > {f_cut:,.0f} or M > {m_cut:,.0f} {REPORTING_CURRENCY})",
      "warning", cust.is_outlier.sum(), len(cust),
      "Excluded from scoring and reported separately" if OUTLIER_ACTION == "exclude" else "Kept")
outliers = cust[cust.is_outlier].sort_values("monetary", ascending=False)
if len(outliers):
    display(Markdown(f"**Outlier customers** ({len(outliers)}) — share of identified net revenue: "
                     f"{outliers.monetary.sum() / cust.monetary.sum():.1%}"))
    display(outliers[["frequency", "monetary", "aov", "recency_days"]].head(10))

### 4.6 L0 report

In [ ]:
dq_df = pd.DataFrame(dq)
dq_df = dq_df[dq_df.rows > 0].copy()
sev_order = {"critical": 0, "warning": 1, "info": 2}
dq_df = dq_df.sort_values(["severity", "rows"], key=lambda s: s.map(sev_order) if s.name == "severity" else -s)
clog_df = pd.DataFrame(clog)

SEV_BG = {"critical": "#fde2e1", "warning": "#fff4d6", "info": "#eef3fb"}
display(Markdown("#### Issues found"))
display(dq_df.style.format({"share": "{:.2%}", "rows": "{:,}"})
        .apply(lambda r: [f"background-color:{SEV_BG[r.severity]}"] * len(r), axis=1).hide(axis="index"))
display(Markdown("#### Cleaning log (transactions)"))
display(clog_df.style.format({c: "{:,}" for c in ["rows_before", "rows_after", "removed"]}).hide(axis="index"))

n_crit = (dq_df.severity == "critical").sum()
summary = (f"**L0 result:** {N0:,} transaction rows received → {len(txi):,} identified rows used "
           f"({len(cust):,} customers). {n_crit} critical, {(dq_df.severity == 'warning').sum()} warning, "
           f"{(dq_df.severity == 'info').sum()} info checks with findings.")
display(Markdown(summary))
if n_crit:
    display(Markdown("⚠️ **Critical issues found** — results below are indicative until the client fixes the export."))

## 5. L1 — Customer segmentation
### 5.1 R, F and M scores
* **R** — quintiles of days since last purchase (5 = most recent)
* **F** — fixed bins on number of purchases (default 1 | 2 | 3 | 4–5 | 6+)
* **M** — quintiles of net revenue in the window after returns, in reporting currency (5 = highest)

Thresholds are saved to `rfm_thresholds.json`; set `FROZEN_THRESHOLDS_PATH` to reuse them in later runs.

In [ ]:
scored = cust[~cust.is_outlier].copy() if OUTLIER_ACTION == "exclude" else cust.copy()

if FROZEN_THRESHOLDS_PATH:
    with open(FROZEN_THRESHOLDS_PATH) as f:
        TH = json.load(f)
    print("Using frozen thresholds from", FROZEN_THRESHOLDS_PATH)
else:
    TH = {
        "recency_days_edges": scored.recency_days.quantile([0.2, 0.4, 0.6, 0.8]).round(0).tolist(),
        "monetary_edges": scored.monetary.quantile([0.2, 0.4, 0.6, 0.8]).round(2).tolist(),
        "frequency_bin_edges": F_BIN_EDGES,
        "reporting_currency": REPORTING_CURRENCY, "snapshot_date": str(SNAP.date()),
        "lookback_months": LOOKBACK_MONTHS,
    }
# R: fewer days = higher score; ties fall into the higher-recency bucket
scored["R"] = 5 - np.searchsorted(TH["recency_days_edges"], scored.recency_days, side="left")
scored["F"] = np.searchsorted(TH["frequency_bin_edges"], scored.frequency, side="right")
scored["M"] = 1 + np.searchsorted(TH["monetary_edges"], scored.monetary, side="right")
scored[["R", "F", "M"]] = scored[["R", "F", "M"]].clip(1, 5).astype(int)

th_table = pd.DataFrame({
    "score": [1, 2, 3, 4, 5],
    "R (days since last purchase)": [f"> {TH['recency_days_edges'][3]:.0f}", f"{TH['recency_days_edges'][2]:.0f}–{TH['recency_days_edges'][3]:.0f}",
                                      f"{TH['recency_days_edges'][1]:.0f}–{TH['recency_days_edges'][2]:.0f}",
                                      f"{TH['recency_days_edges'][0]:.0f}–{TH['recency_days_edges'][1]:.0f}", f"≤ {TH['recency_days_edges'][0]:.0f}"],
    "F (purchases)": ["1", "2", "3", "4–5", "6+"] if TH["frequency_bin_edges"] == [1, 2, 3, 4, 6] else
                     [f"≥ {e}" for e in TH["frequency_bin_edges"]],
    f"M (net revenue, {REPORTING_CURRENCY})": [f"≤ {TH['monetary_edges'][0]:,.0f}", f"{TH['monetary_edges'][0]:,.0f}–{TH['monetary_edges'][1]:,.0f}",
                                               f"{TH['monetary_edges'][1]:,.0f}–{TH['monetary_edges'][2]:,.0f}",
                                               f"{TH['monetary_edges'][2]:,.0f}–{TH['monetary_edges'][3]:,.0f}", f"> {TH['monetary_edges'][3]:,.0f}"],
})
display(th_table.style.hide(axis="index"))
display(pd.DataFrame({s: scored[s].value_counts().sort_index() for s in "RFM"}).rename_axis("score")
        .T.style.format("{:,}"))

### 5.2 Segment assignment

In [ ]:
scored["segment"] = None
for name, rule in SEGMENT_RULES:
    free = scored.segment.isna()
    hit = scored[free].eval(rule)
    scored.loc[hit[hit].index, "segment"] = name
unassigned = scored.segment.isna().sum()
if unassigned:
    print(f"{unassigned:,} customers matched no rule → 'Unassigned'. Add a catch-all rule.")
    scored["segment"] = scored.segment.fillna("Unassigned")
SEG_ORDER = [s for s, _ in SEGMENT_RULES if s in scored.segment.unique()] + (["Unassigned"] if unassigned else [])

seg_def = pd.DataFrame([dict(order=i + 1, segment=s, rule=rule, description=SEGMENT_INFO.get(s, ("", ""))[0],
                             suggested_action=SEGMENT_INFO.get(s, ("", ""))[1])
                        for i, (s, rule) in enumerate(SEGMENT_RULES)])
display(Markdown("#### Segment definitions (first matching rule wins)"))
display(seg_def.style.hide(axis="index"))

### 5.3 Segment KPIs

In [ ]:
tot_c, tot_m = len(scored), scored.monetary.sum()
g = scored.groupby("segment")
kpi = pd.DataFrame({
    "customers": g.size(),
    "customer_share": g.size() / tot_c,
    "net_revenue": g.monetary.sum(),
    "revenue_share": g.monetary.sum() / tot_m,
    "revenue_per_customer": g.monetary.mean(),
    "aov": g.gross_revenue.sum() / g.frequency.sum(),
    "avg_purchases": g.frequency.mean(),
    "median_days_since_purchase": g.recency_days.median(),
    "return_rate": -g.returns.sum() / g.gross_revenue.sum(),
}).reindex(SEG_ORDER)
if cust.loyalty_member.notna().any():
    kpi["loyalty_share"] = g.loyalty_member.apply(lambda s: s.fillna(False).astype(bool).mean()).reindex(SEG_ORDER)
if cust.marketing_consent.notna().any():
    kpi["consent_share"] = g.marketing_consent.apply(lambda s: s.fillna(False).astype(bool).mean()).reindex(SEG_ORDER)
total = pd.DataFrame({"customers": tot_c, "customer_share": 1.0, "net_revenue": tot_m, "revenue_share": 1.0,
                      "revenue_per_customer": scored.monetary.mean(),
                      "aov": scored.gross_revenue.sum() / scored.frequency.sum(),
                      "avg_purchases": scored.frequency.mean(),
                      "median_days_since_purchase": scored.recency_days.median(),
                      "return_rate": -scored.returns.sum() / scored.gross_revenue.sum()}, index=["Total"])
for c, src in [("loyalty_share", "loyalty_member"), ("consent_share", "marketing_consent")]:
    if c in kpi:
        total[c] = scored[src].fillna(False).astype(bool).mean()
kpi_all = pd.concat([kpi, total])
fmt = {"customers": "{:,.0f}", "customer_share": "{:.1%}", "net_revenue": "{:,.0f}", "revenue_share": "{:.1%}",
       "revenue_per_customer": "{:,.0f}", "aov": "{:,.0f}", "avg_purchases": "{:.1f}",
       "median_days_since_purchase": "{:.0f}", "return_rate": "{:.1%}", "loyalty_share": "{:.0%}", "consent_share": "{:.0%}"}
display(Markdown(f"Monetary values in **{REPORTING_CURRENCY}**, net of returns, excl. VAT."))
display(kpi_all.style.format({k: v for k, v in fmt.items() if k in kpi_all})
        .bar(subset=["revenue_share"], color="#b7d3f6", vmin=0, vmax=kpi.revenue_share.max())
        .set_properties(subset=pd.IndexSlice[["Total"], :], **{"font-weight": "bold"}))

### 5.4 Charts

In [ ]:
# 1) Share of customers vs share of revenue per segment
k = kpi.iloc[::-1]
fig = go.Figure()
fig.add_bar(y=k.index, x=k.customer_share, name="Share of customers", orientation="h", marker_color=C["blue"],
            text=[f"{v:.0%}" for v in k.customer_share], textposition="outside",
            hovertemplate="%{y}<br>Customers: %{x:.1%}<extra></extra>")
fig.add_bar(y=k.index, x=k.revenue_share, name="Share of revenue", orientation="h", marker_color=C["orange"],
            text=[f"{v:.0%}" for v in k.revenue_share], textposition="outside",
            hovertemplate="%{y}<br>Revenue: %{x:.1%}<extra></extra>")
fig.update_layout(barmode="group", bargap=0.3, bargroupgap=0.08)
fig.update_xaxes(tickformat=".0%", range=[0, max(k.customer_share.max(), k.revenue_share.max()) * 1.15])
style(fig, "Segments: share of customers vs share of revenue", height=460).show()

In [ ]:
# 2) R x F heatmap: customers, with revenue per customer in hover
hm_c = scored.pivot_table(index="F", columns="R", values="frequency", aggfunc="size").reindex(index=range(5, 0, -1), columns=range(1, 6))
hm_m = scored.pivot_table(index="F", columns="R", values="monetary", aggfunc="mean").reindex(index=range(5, 0, -1), columns=range(1, 6))
fig = go.Figure(go.Heatmap(
    z=hm_c.values, x=[f"R{c}" for c in hm_c.columns], y=[f"F{i}" for i in hm_c.index],
    colorscale=[[i / (len(SEQ_BLUE) - 1), c] for i, c in enumerate(SEQ_BLUE)],
    customdata=hm_m.values, xgap=2, ygap=2, colorbar=dict(title="Customers"),
    text=np.where(np.isnan(hm_c.values), "", np.vectorize(lambda v: f"{v:,.0f}")(np.nan_to_num(hm_c.values))),
    texttemplate="%{text}", textfont=dict(size=11),
    hovertemplate="%{x} · %{y}<br>Customers: %{z:,}<br>Revenue / customer: %{customdata:,.0f} " + REPORTING_CURRENCY + "<extra></extra>"))
fig.update_xaxes(title="Recency score (5 = most recent)", showgrid=False)
fig.update_yaxes(title="Frequency score", showgrid=False)
style(fig, "Customers by Recency and Frequency score", height=440).show()

In [ ]:
# 3) Revenue concentration
srt = scored.monetary.clip(lower=0).sort_values(ascending=False).values
cum = np.cumsum(srt) / srt.sum()
x = np.arange(1, len(srt) + 1) / len(srt)
idx = np.linspace(0, len(srt) - 1, min(len(srt), 500)).astype(int)
top20 = cum[int(len(srt) * 0.2) - 1]
fig = go.Figure(go.Scatter(x=x[idx], y=cum[idx], mode="lines", line=dict(color=C["blue"], width=2),
                           hovertemplate="Top %{x:.0%} of customers → %{y:.0%} of revenue<extra></extra>", name=""))
fig.add_trace(go.Scatter(x=[0.2], y=[top20], mode="markers+text", marker=dict(size=9, color=C["blue"], line=dict(color="white", width=2)),
                         text=[f"  Top 20% → {top20:.0%} of revenue"], textposition="middle right", hoverinfo="skip", showlegend=False))
fig.update_xaxes(title="Share of customers (ranked by net revenue)", tickformat=".0%", range=[0, 1])
fig.update_yaxes(title="Cumulative share of revenue", tickformat=".0%", range=[0, 1.02])
fig.update_layout(showlegend=False)
style(fig, "Revenue concentration", height=420).show()

In [ ]:
# 4) Distributions of the three RFM inputs
fig = make_subplots(rows=1, cols=3, subplot_titles=("Days since last purchase", "Purchases in window",
                                                    f"Net revenue per customer ({REPORTING_CURRENCY}, log)"))
fig.add_histogram(x=scored.recency_days, nbinsx=48, marker_color=C["blue"], row=1, col=1,
                  hovertemplate="%{x} days<br>%{y:,} customers<extra></extra>")
fv = scored.frequency.clip(upper=15)
fig.add_histogram(x=fv, xbins=dict(start=0.5, end=15.5, size=1), marker_color=C["blue"], row=1, col=2,
                  hovertemplate="%{x} purchases (15 = 15+)<br>%{y:,} customers<extra></extra>")
pos_m = scored.monetary[scored.monetary > 0]
fig.add_histogram(x=np.log10(pos_m), nbinsx=50, marker_color=C["blue"], row=1, col=3,
                  hovertemplate="%{y:,} customers<extra></extra>")
fig.update_xaxes(tickvals=[1, 2, 3, 4, 5], ticktext=["10", "100", "1k", "10k", "100k"], row=1, col=3)
fig.update_layout(showlegend=False, bargap=0.05)
n_nonpos = int((scored.monetary <= 0).sum())
title = "Distribution of R, F and M inputs" + (f"  ·  {n_nonpos:,} customers with net revenue ≤ 0 (fully returned) not shown in M" if n_nonpos else "")
style(fig, title, height=360).show()

In [ ]:
# 5) Segment map: average recency vs average purchases, bubble = revenue
fig = go.Figure(go.Scatter(
    x=kpi.median_days_since_purchase, y=kpi.avg_purchases, mode="markers+text", text=kpi.index,
    textposition=["bottom center" if s == "Reactivated" else "top center" for s in kpi.index], customdata=np.c_[kpi.customers, kpi.revenue_share, kpi.revenue_per_customer],
    marker=dict(size=np.sqrt(kpi.net_revenue / kpi.net_revenue.max()) * 70 + 8, color=C["blue"], opacity=0.75,
                line=dict(color="white", width=2)),
    hovertemplate="<b>%{text}</b><br>Median days since purchase: %{x:.0f}<br>Avg purchases: %{y:.1f}"
                  "<br>Customers: %{customdata[0]:,.0f}<br>Revenue share: %{customdata[1]:.1%}"
                  "<br>Revenue / customer: %{customdata[2]:,.0f}<extra></extra>"))
fig.update_xaxes(title="Median days since last purchase", autorange="reversed")
fig.update_yaxes(title="Average purchases in window", rangemode="tozero")
style(fig, "Segment map (bubble size = net revenue)", height=480).show()

### 5.5 Key insights (generated)

In [ ]:
def seg(name, col):
    return kpi.at[name, col] if name in kpi.index else np.nan

ins = []
ins.append(f"**{tot_c:,} customers** generated **{tot_m:,.0f} {REPORTING_CURRENCY}** net revenue in the last "
           f"{LOOKBACK_MONTHS} months; average {scored.frequency.mean():.1f} purchases and "
           f"{scored.monetary.mean():,.0f} {REPORTING_CURRENCY} per customer.")
ins.append(f"**Concentration:** the top 20% of customers account for **{top20:.0%}** of revenue.")
repeat = (scored.frequency >= 2).mean()
ins.append(f"**Repeat rate:** {repeat:.0%} of customers bought more than once in the window.")
if "Champions" in kpi.index:
    ins.append(f"**Champions** are {seg('Champions','customer_share'):.0%} of customers but "
               f"{seg('Champions','revenue_share'):.0%} of revenue, spending "
               f"{seg('Champions','revenue_per_customer') / scored.monetary.mean():.1f}× the average customer.")
hv = [s for s in ["Champions", "Loyal High Value"] if s in kpi.index]
if hv:
    ins.append(f"**High-value core** ({' + '.join(hv)}): {kpi.loc[hv,'customer_share'].sum():.0%} of customers, "
               f"{kpi.loc[hv,'revenue_share'].sum():.0%} of revenue.")
if "At Risk High Value" in kpi.index:
    ins.append(f"**At Risk High Value:** {seg('At Risk High Value','customers'):,.0f} customers who produced "
               f"{seg('At Risk High Value','net_revenue'):,.0f} {REPORTING_CURRENCY} "
               f"({seg('At Risk High Value','revenue_share'):.0%} of revenue) have not bought for a median of "
               f"{seg('At Risk High Value','median_days_since_purchase'):.0f} days — the main win-back target.")
if "New Customers" in kpi.index:
    ins.append(f"**New Customers:** {seg('New Customers','customers'):,.0f} ({seg('New Customers','customer_share'):.0%}) "
               f"made their first-ever purchase in the last ~{TH['recency_days_edges'][1]:.0f} days; converting them to a "
               f"second order is the cheapest growth lever.")
if "Reactivated" in kpi.index:
    ins.append(f"**Reactivated:** {seg('Reactivated','customers'):,.0f} recent single-order customers had bought before the "
               f"window — without the customers table they would have been counted as new.")
if "Hibernating" in kpi.index:
    ins.append(f"**Hibernating:** {seg('Hibernating','customer_share'):.0%} of customers but only "
               f"{seg('Hibernating','revenue_share'):.0%} of revenue; consider low-cost reactivation or suppression.")
rr = kpi.return_rate.dropna()
if len(rr):
    ins.append(f"**Returns:** overall return rate {total.return_rate.iat[0]:.1%}; highest in **{rr.idxmax()}** ({rr.max():.1%}).")
if "consent_share" in kpi:
    act = scored.marketing_consent.fillna(False).astype(bool)
    ins.append(f"**Activation:** {act.mean():.0%} of scored customers have marketing consent "
               f"({act.sum():,} addressable via email / Customer Match / Meta).")
if cust.is_outlier.any():
    ins.append(f"**Outliers:** {cust.is_outlier.sum()} customer(s) flagged as outlier/B2B "
               f"({'excluded' if OUTLIER_ACTION == 'exclude' else 'kept'}), "
               f"{outliers.monetary.sum():,.0f} {REPORTING_CURRENCY} of revenue.")
cov_key = "Retail purchases linked to a customer"
if cov_key in coverage:
    ins.append(f"**Coverage caveat:** {coverage[cov_key]:.0%} of retail purchases are linked to a customer; "
               f"retail behaviour of anonymous shoppers is not reflected.")
display(Markdown("\n".join(f"- {s}" for s in ins)))

## 6. Export
Writes the per-customer segment table (with snapshot date, for L5 comparisons later), KPIs, definitions,
L0 report and thresholds.

In [ ]:
if OUTPUT_FOLDER is None:
    base = DRIVE_FOLDER if SOURCE == "drive" else LOCAL_FOLDER if SOURCE == "local" else "."
    OUTPUT_FOLDER = os.path.join(base, "output", f"rfm_{SNAP.date()}")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

out = scored.reset_index()[["customer_key", "recency_days", "frequency", "monetary", "gross_revenue", "returns", "aov",
                            "R", "F", "M", "segment", "first_purchase_date", "has_prior_history",
                            "loyalty_member", "marketing_consent"]].copy()
out["rfm_code"] = out.R.astype(str) + out.F.astype(str) + out.M.astype(str)
out["snapshot_date"] = str(SNAP.date())
out["client"] = CLIENT_NAME
if OUTLIER_ACTION == "exclude" and len(outliers):
    o = outliers.reset_index()[["customer_key", "recency_days", "frequency", "monetary", "gross_revenue", "returns", "aov"]]
    o["segment"] = "Outlier (excluded)"; o["snapshot_date"] = str(SNAP.date()); o["client"] = CLIENT_NAME
    out = pd.concat([out, o], ignore_index=True)

files_out = {
    "customer_segments.csv": out,
    "segment_kpis.csv": kpi_all.rename_axis("segment").reset_index(),
    "segment_definitions.csv": seg_def,
    "l0_dq_report.csv": dq_df,
    "l0_cleaning_log.csv": clog_df,
}
for fn, df in files_out.items():
    df.to_csv(os.path.join(OUTPUT_FOLDER, fn), index=False)
with open(os.path.join(OUTPUT_FOLDER, "rfm_thresholds.json"), "w") as f:
    json.dump(TH, f, indent=2)
meta = dict(client=CLIENT_NAME, run_at=datetime.now().isoformat(timespec="seconds"), source=SOURCE,
            snapshot_date=str(SNAP.date()), window_start=str(WSTART.date()), lookback_months=LOOKBACK_MONTHS,
            reporting_currency=REPORTING_CURRENCY, outlier_action=OUTLIER_ACTION, customers_scored=int(tot_c),
            segment_rules=SEGMENT_RULES, coverage={k: round(float(v), 4) for k, v in coverage.items()})
with open(os.path.join(OUTPUT_FOLDER, "run_metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)

if WRITE_TO_BIGQUERY:
    from google.cloud import bigquery
    if IN_COLAB:
        from google.colab import auth; auth.authenticate_user()
    bq = bigquery.Client(project=BQ_BILLING_PROJECT)
    job = bq.load_table_from_dataframe(out.astype({"first_purchase_date": "datetime64[ns]"}), BQ_OUTPUT_TABLE,
                                       job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND"))
    job.result(); print("Appended to", BQ_OUTPUT_TABLE)

print("Written to", OUTPUT_FOLDER)
for fn in sorted(os.listdir(OUTPUT_FOLDER)):
    print("  ", fn)